# 02. 그리드 인코딩, 디코딩, 클래스별 NMS

## 학습 목표

객체 중심을 담당 셀에 배정하고 YOLO v1 텐서를 구성한 뒤, 후보 박스를 다시 이미지 좌표로 복원하고 클래스별 NMS로 중복을 줄입니다.

## 실행 방법

`python -m pip install jupyter numpy` 후 Jupyter에서 위에서 아래로 실행합니다. 이 실습의 채널 배치는 학습 편의를 위해 `[클래스 C개, 박스1의 x/y/w/h/conf, 박스2 ...]`로 고정합니다.

In [ ]:
import numpy as np  # 행렬 형태의 그리드 텐서와 벡터 연산을 위해 NumPy를 가져옵니다.

GRID_SIZE = 7  # YOLO v1 논문과 같은 S=7 그리드 크기를 상수로 둡니다.
BOXES_PER_CELL = 2  # 한 셀이 예측하는 박스 수 B=2를 사용합니다.
CLASS_COUNT = 3  # 계산을 눈으로 확인하기 쉽도록 VOC 20개 대신 장난감 클래스 3개를 사용합니다.
CHANNEL_COUNT = CLASS_COUNT + BOXES_PER_CELL * 5  # 클래스 C개와 박스마다 x,y,w,h,confidence 다섯 값을 합칩니다.

print(f"실습 텐서: {GRID_SIZE}×{GRID_SIZE}×{CHANNEL_COUNT}")  # 설정에서 계산된 텐서 크기를 출력합니다.
print(f"VOC 설정이라면: 7×7×{20 + 2 * 5}")  # C=20인 논문 설정은 7×7×30임을 비교합니다.

## 1. 객체를 담당 그리드 셀에 배정하기

YOLO v1에서 객체 중심이 들어간 셀이 그 객체를 담당합니다. `cx=1.0`처럼 오른쪽 경계에 정확히 놓인 입력은 정수 변환 시 7이 되므로 마지막 유효 인덱스 6으로 제한합니다.

In [ ]:
def assign_grid_cell(cx, cy, grid_size=GRID_SIZE):  # 정규화 중심 좌표를 행, 열, 셀 내부 좌표로 변환합니다.
    if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):  # chained comparison과 and로 두 좌표의 유효 범위를 검사합니다.
        raise ValueError("cx와 cy는 0 이상 1 이하여야 합니다.")  # 이미지 바깥 중심을 즉시 거부합니다.

    column = min(int(cx * grid_size), grid_size - 1)  # x를 셀 열로 바꾸고 오른쪽 경계는 마지막 열로 제한합니다.
    row = min(int(cy * grid_size), grid_size - 1)  # y를 셀 행으로 바꾸고 아래쪽 경계는 마지막 행으로 제한합니다.
    local_x = cx * grid_size - column  # 이미지 좌표를 해당 셀 왼쪽 기준의 상대 x로 변환합니다.
    local_y = cy * grid_size - row  # 이미지 좌표를 해당 셀 위쪽 기준의 상대 y로 변환합니다.
    return row, column, local_x, local_y  # 여러 결과를 튜플로 묶어 반환합니다.


row, column, local_x, local_y = assign_grid_cell(0.52, 0.38)  # 예제 중심을 담당 셀로 변환하며 튜플 언패킹을 사용합니다.
print(row, column, local_x, local_y)  # 결과는 2행 3열이며 셀 내부 좌표는 각각 약 0.64, 0.66입니다.

assert (row, column) == (2, 3)  # 중심 좌표가 기대한 셀에 배정됐는지 확인합니다.
assert assign_grid_cell(1.0, 1.0)[:2] == (6, 6)  # 정확한 오른쪽 아래 경계도 배열 밖으로 나가지 않아야 합니다.

## 2. 한 객체를 텐서에 인코딩하고 다시 디코딩하기

YOLO v1의 `x, y`는 셀 경계에 상대적이고 `w, h`는 전체 이미지에 상대적입니다. 아래 예제에서는 두 박스 슬롯에 같은 기하 정보를 넣되 신뢰도만 다르게 주어 후속 억제 과정을 관찰합니다.

In [ ]:
def encode_single_object(cx, cy, width, height, class_index):  # 객체 하나를 빈 YOLO 형식 텐서에 기록합니다.
    if not (0 <= class_index < CLASS_COUNT):  # 클래스 인덱스가 0부터 C-1 사이인지 확인합니다.
        raise ValueError("class_index가 클래스 범위를 벗어났습니다.")  # 잘못된 원-핫 위치 접근을 예방합니다.
    if width <= 0.0 or height <= 0.0:  # 폭과 높이는 양수여야 하므로 논리합 or로 검사합니다.
        raise ValueError("width와 height는 양수여야 합니다.")  # 무효한 박스를 텐서에 넣지 않습니다.

    tensor = np.zeros((GRID_SIZE, GRID_SIZE, CHANNEL_COUNT), dtype=np.float64)  # 모든 예측이 0인 3차원 텐서를 만듭니다.
    row, column, local_x, local_y = assign_grid_cell(cx, cy)  # 중심 좌표로 담당 셀과 셀 내부 좌표를 구합니다.
    tensor[row, column, class_index] = 1.0  # 담당 셀의 클래스 벡터를 원-핫 방식으로 표시합니다.

    for box_index in range(BOXES_PER_CELL):  # range(2)로 두 박스 슬롯을 0, 1 순서로 반복합니다.
        start = CLASS_COUNT + box_index * 5  # 각 박스의 다섯 채널이 시작되는 인덱스를 계산합니다.
        confidence = 0.90 - box_index * 0.15  # 실습을 위해 첫 박스 0.90, 둘째 박스 0.75를 부여합니다.
        tensor[row, column, start : start + 5] = [local_x, local_y, width, height, confidence]  # 슬라이스에 다섯 값을 한 번에 기록합니다.

    return tensor, (row, column)  # 텐서와 담당 셀 위치를 함께 반환합니다.


encoded, responsible_cell = encode_single_object(0.52, 0.38, 0.24, 0.30, class_index=1)  # 클래스 1 객체 하나를 인코딩합니다.
print("담당 셀:", responsible_cell)  # 객체 중심을 포함한 행과 열을 출력합니다.
print("담당 셀 벡터:", encoded[responsible_cell])  # 튜플 인덱싱으로 그 셀의 전체 채널을 확인합니다.

assert encoded.shape == (7, 7, 13)  # C=3, B=2이면 마지막 축은 3+2×5=13이어야 합니다.
assert encoded[responsible_cell][1] == 1.0  # 클래스 1의 원-핫 값이 정확히 설정됐는지 확인합니다.

In [ ]:
def decode_cell_box(tensor, row, column, box_index):  # 특정 셀의 특정 박스를 이미지 전체 기준 중심 좌표로 복원합니다.
    if not (0 <= box_index < BOXES_PER_CELL):  # 요청한 박스 슬롯이 존재하는 범위인지 검사합니다.
        raise ValueError("box_index가 박스 슬롯 범위를 벗어났습니다.")  # 배열 범위 밖 접근 대신 의미 있는 오류를 냅니다.

    start = CLASS_COUNT + box_index * 5  # 선택한 박스 슬롯의 첫 채널 위치를 계산합니다.
    local_x, local_y, width, height, confidence = tensor[row, column, start : start + 5]  # 다섯 값을 변수별로 언패킹합니다.
    cx = (column + local_x) / GRID_SIZE  # 열 번호와 셀 내부 x를 합친 뒤 S로 나눠 이미지 정규화 x를 복원합니다.
    cy = (row + local_y) / GRID_SIZE  # 행 번호와 셀 내부 y를 합친 뒤 S로 나눠 이미지 정규화 y를 복원합니다.
    return np.array([cx, cy, width, height]), float(confidence)  # 박스 배열과 신뢰도를 서로 다른 값으로 반환합니다.


decoded_box, decoded_confidence = decode_cell_box(encoded, *responsible_cell, box_index=0)  # *는 (row,column)을 두 위치 인수로 펼칩니다.
print("복원 박스:", decoded_box)  # 인코딩 전 [0.52,0.38,0.24,0.30]과 같아야 합니다.
print("복원 신뢰도:", decoded_confidence)  # 첫 박스에 넣었던 0.90이 나와야 합니다.

assert np.allclose(decoded_box, [0.52, 0.38, 0.24, 0.30])  # 인코딩과 디코딩의 왕복 결과를 검증합니다.
assert np.isclose(decoded_confidence, 0.90)  # 부동소수점 오차를 고려해 신뢰도를 비교합니다.

## 3. 클래스별 비최대 억제(NMS)

NMS는 점수가 가장 높은 박스를 남기고, 같은 클래스에서 그 박스와 IoU가 임계값보다 큰 낮은 점수 박스를 제거합니다. 서로 다른 클래스끼리는 억제하지 않는 것이 이 구현의 중요한 계약입니다.

In [ ]:
def iou_one_to_many(reference_box, candidate_boxes, epsilon=1e-12):  # 기준 박스 하나와 후보 여러 개의 IoU를 벡터로 계산합니다.
    reference_box = np.asarray(reference_box, dtype=np.float64)  # 기준 박스를 실수 배열로 변환합니다.
    candidate_boxes = np.asarray(candidate_boxes, dtype=np.float64)  # 후보 목록도 실수 배열로 변환합니다.
    top_left = np.maximum(reference_box[:2], candidate_boxes[:, :2])  # 브로드캐스팅으로 모든 교집합 시작점을 계산합니다.
    bottom_right = np.minimum(reference_box[2:], candidate_boxes[:, 2:])  # 모든 교집합 끝점을 계산합니다.
    intersection_size = np.maximum(bottom_right - top_left, 0.0)  # 겹치지 않는 축의 길이를 0으로 제한합니다.
    intersection = intersection_size[:, 0] * intersection_size[:, 1]  # 후보별 교집합 넓이를 구합니다.
    reference_area = np.prod(np.maximum(reference_box[2:] - reference_box[:2], 0.0))  # prod로 기준 폭과 높이를 곱합니다.
    candidate_areas = np.prod(np.maximum(candidate_boxes[:, 2:] - candidate_boxes[:, :2], 0.0), axis=1)  # axis=1로 후보별 넓이를 구합니다.
    union = reference_area + candidate_areas - intersection  # 합집합 넓이는 두 넓이 합에서 교집합을 뺀 값입니다.
    return intersection / np.maximum(union, epsilon)  # 0 나눗셈을 막은 후보별 IoU를 반환합니다.


def class_aware_nms(boxes, scores, class_ids, iou_threshold=0.5):  # 클래스마다 독립적으로 NMS를 수행합니다.
    boxes = np.asarray(boxes, dtype=np.float64)  # 박스 목록을 (N,4) 실수 배열로 통일합니다.
    scores = np.asarray(scores, dtype=np.float64)  # 점수 목록을 실수 배열로 통일합니다.
    class_ids = np.asarray(class_ids, dtype=np.int64)  # 클래스 ID는 비교에 적합한 정수 배열로 통일합니다.
    kept_indices = []  # 최종적으로 남길 원래 박스 인덱스를 저장할 빈 리스트입니다.

    for class_id in np.unique(class_ids):  # unique로 중복을 제거한 각 클래스에 대해 반복합니다.
        class_indices = np.flatnonzero(class_ids == class_id)  # 현재 클래스인 위치만 1차원 인덱스로 얻습니다.
        order = class_indices[np.argsort(scores[class_indices])[::-1]]  # 점수를 오름차순 정렬한 뒤 [::-1]로 뒤집어 내림차순으로 만듭니다.

        while order.size > 0:  # 아직 검사할 후보가 남아 있는 동안 반복합니다.
            best_index = int(order[0])  # 현재 최고 점수 박스의 원래 인덱스를 선택합니다.
            kept_indices.append(best_index)  # 최고 점수 박스는 결과에 보존합니다.
            if order.size == 1:  # 비교할 나머지 박스가 없으면 현재 클래스 처리를 끝냅니다.
                break  # break는 가장 가까운 while 반복문을 종료합니다.
            remaining = order[1:]  # 최고 점수를 제외한 나머지 후보를 선택합니다.
            overlaps = iou_one_to_many(boxes[best_index], boxes[remaining])  # 최고 박스와 나머지의 IoU를 한 번에 구합니다.
            order = remaining[overlaps <= iou_threshold]  # 임계값 이하로 충분히 다른 박스만 다음 반복에 남깁니다.

    return np.array(sorted(kept_indices, key=lambda index: scores[index], reverse=True), dtype=np.int64)  # 전체 결과를 점수 내림차순으로 정렬해 반환합니다.


candidate_boxes = np.array(  # NMS 동작을 확인할 네 후보 박스를 정의합니다.
    [[0.10, 0.10, 0.50, 0.50], [0.12, 0.12, 0.52, 0.52], [0.60, 0.60, 0.90, 0.90], [0.11, 0.11, 0.51, 0.51]],  # 앞의 두 박스와 마지막 박스는 크게 겹칩니다.
    dtype=np.float64,  # 좌표를 실수로 저장합니다.
)
candidate_scores = np.array([0.95, 0.80, 0.70, 0.75])  # 박스별 클래스 점수를 정의합니다.
candidate_classes = np.array([0, 0, 0, 1])  # 마지막 박스는 겹치더라도 다른 클래스이므로 유지돼야 합니다.
kept = class_aware_nms(candidate_boxes, candidate_scores, candidate_classes, iou_threshold=0.5)  # IoU 0.5를 넘는 동클래스 중복을 억제합니다.

print("유지된 인덱스:", kept)  # 기대 결과는 점수 순서의 [0, 3, 2]입니다.
assert kept.tolist() == [0, 3, 2]  # 동클래스 중복 1은 제거되고 타 클래스 3은 남는지 확인합니다.

## 직접 해볼 과제

1. NMS 임계값을 0.3, 0.7로 바꿔 유지되는 박스가 어떻게 달라지는지 확인하세요.
2. 객체 두 개의 중심이 같은 셀에 들어가는 사례를 만들고 YOLO v1 그리드 설계의 한계를 설명하세요.
3. 실제 제품에서는 모델 출력의 활성화 함수와 좌표 정의가 구현마다 다를 수 있습니다. 배포 모델의 명세를 이 노트북의 채널 계약과 대조하세요.